# Testing with Dask

In [1]:
# setup imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path().resolve().parents[2]  # pas aan als notebook dieper/dichter zit
sys.path.append(str(PROJECT_ROOT))

In [2]:
import warnings

import seaborn as sns
from dask import delayed
from dask.distributed import Client, progress
from ewatercycle.observation.grdc import get_grdc_data

from src.forcing import load_lumped_forcing
from src.models import *
from src.paths import *

warnings.filterwarnings("ignore", category=UserWarning)

/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498


In [3]:
ERA5_forcing_loaded = load_lumped_forcing("Kerki_GRDC", "ERA5", "1959-1963")
theta_test = np.mean(generate_HBV_parameters(100), axis=0)

grdc_station = get_grdc_data(
    2617110, "1959-01-02T00:00Z", "1963-12-31T00:00Z", data_home=GRDC / "Daily"
)

q_obs = grdc_station["streamflow"]
years = q_obs["time"].dt.year.values
shape_name = "Kerki_GRDC"

In [4]:
# CMA-ES test setup
x0_norm = scale(0.5 * (p_min + p_max))  # start in the middle
sigma0 = 0.1  # step size in normalized space

# Define 20 unique seeds
seeds = [
    180988,
    214025,
    987654,
    123456,
    654321,
    111222,
    333444,
    555666,
    777888,
    999000,
    112233,
    445566,
    778899,
    101010,
    202020,
    303030,
    404040,
    505050,
    606060,
    707070,
]

# results = []

# for s in seeds:
#     res = run_cma(
#         seed=s,
#         x0_norm=x0_norm,
#         sigma0=sigma0,
#         objective_fn=objective_safe,
#         popsize=18,
#         maxfevals=360,
#         save_path=f"results/result_seed_{s}.pkl",
#     )
#     results.append(res)

In [5]:
# Create a Dask client with 3 workers, 1 thread each, dashboard auto-assigned
client = Client(n_workers=3, threads_per_worker=1, dashboard_address=":0")

# Display client info (clickable dashboard link appears in Jupyter)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:36605/status,
Dashboard: http://127.0.0.1:36605/status,Workers: 3
Total threads: 3,Total memory: 31.34 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42483,Workers: 3
Dashboard: http://127.0.0.1:36605/status,Total threads: 3
Started: Just now,Total memory: 31.34 GiB
Comm: tcp://127.0.0.1:36317,Total threads: 1
Dashboard: http://127.0.0.1:46653/status,Memory: 10.45 GiB
Nanny: tcp://127.0.0.1:36085,


In [6]:
client = Client()
futures = client.futures  # all submitted tasks

# Print active, pending, finished
for f in futures:
    print(f.key, f.status)

In [7]:
from dask.distributed import Client

# Assuming you created a client earlier
client.close()  # closes client and shuts down local cluster/workers

In [8]:
import warnings

from dask.distributed import Client, default_client


def close_dask():
    try:
        # Try to get the current default client
        client = default_client()
        print(
            f"Closing existing Dask client with {len(client.scheduler_info()['workers'])} workers..."
        )
        client.close()
        print("Dask client closed successfully.")
    except ValueError:
        # No active client
        print("No active Dask client found.")
    except Exception as e:
        warnings.warn(f"Error closing Dask client: {e}")


# Run the cleanup
close_dask()

Closing existing Dask client with 3 workers...
Dask client closed successfully.


In [8]:
# from dask import delayed, compute

# objective_fn = make_objective_safe(ERA5_forcing_loaded, q_obs.values, shape_name)
# number_runs = 10
# test_seeds = seeds[:number_runs]

# tasks = [delayed(run_cma)(
#     seed=s,
#     x0_norm=x0_norm,
#     sigma0=sigma0,
#     objective_fn=objective_fn,
#     popsize=10,
#     maxfevals=40,
#     save_folder="results_2/"
# ) for s in test_seeds]

# results = compute(*tasks)
# #print(results)

In [9]:
objective_fn = make_objective_safe(ERA5_forcing_loaded, q_obs.values, shape_name)
number_runs = 18
test_seeds = seeds[:number_runs]

tasks = [
    delayed(run_cma)(
        cma_seed=s,
        x0_norm=x0_norm,
        sigma0=sigma0,
        objective_fn=objective_fn,
        popsize=16,
        maxfevals=330,
        save_folder="results_4/",
    )
    for s in test_seeds
]

# Submit tasks to the cluster
futures = client.compute(tasks)

# Optional: see a live progress bar
progress(futures)

# Gather results when done
results = client.gather(futures)

/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498
/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498
/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498


(8_w,16)-aCMA-ES (mu_w=4.8,w_1=32%) in dimension 9 (seed=505050, Tue Jan 27 13:35:32 2026)
(8_w,16)-aCMA-ES (mu_w=4.8,w_1=32%) in dimension 9 (seed=180988, Tue Jan 27 13:35:32 2026)
(8_w,16)-aCMA-ES (mu_w=4.8,w_1=32%) in dimension 9 (seed=654321, Tue Jan 27 13:35:32 2026)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     16 4.004732786567402e-01 1.0e+00 1.08e-01  1e-01  1e-01 2:11.5
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     16 3.303692023026911e-01 1.0e+00 1.09e-01  1e-01  1e-01 2:15.7
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     16 3.303692023026911e-01 1.0e+00 1.11e-01  1e-01  1e-01 2:15.9
    2     32 1.977178796785223e-01 1.4e+00 1.18e-01  1e-01  1e-01 4:22.9
    2     32 3.776876931077315e-01 1.4e+00 1.15e-01  1e-01  1e-01 4:30.8
    2     32 2.216255006982698e-01 1.3e+00 1.28e-01  1e-01  1e-01 4:31.4
    3     48 2.045341921706797e-01 1.4e+00 1.35e-01  1e-01  2e-01 6:37.7


In [ ]:
# Convert results (list of dicts) to a nice table
df_results = pd.DataFrame(results)

# Optional: reorder columns
df_results = df_results[["seed", "best_f", "best_x", "nfev"]]

# Display nicely
df_results

In [ ]:
import pickle
from pathlib import Path

results_folder = Path("results_2")
result_files = list(results_folder.glob("result_seed_*.pkl"))

loaded_results = []
for f in result_files:
    with open(f, "rb") as file:
        res = pickle.load(file)
        loaded_results.append(res)

# Example: print best objective per seed
for r in loaded_results:
    print(f"Seed {r['seed']}: best_f = {r['best_f']}, nfev = {r['nfev']}")

In [ ]:
best_params_list = []

for run_idx, run in enumerate(results):
    seed = run["seed"]
    history = run["history"]

    # find the best evaluation by objective value
    best_idx = np.argmin(history["objective"])

    best_theta_phys = history["theta_phys"][best_idx]
    best_obj = history["objective"][best_idx]
    best_nse = history["nse"][best_idx]
    best_kge = history["kge"][best_idx]
    best_vol_err = history["vol_err"][best_idx]

    row = {
        "seed": seed,
        "best_eval": best_idx + 1,
        "objective": best_obj,
        "nse": best_nse,
        "kge": best_kge,
        "vol_err": best_vol_err,
    }

    # add each parameter with its proper name
    for i, th in enumerate(best_theta_phys):
        row[parameter_names[i]] = th

    best_params_list.append(row)

# create DataFrame
best_params_df = pd.DataFrame(best_params_list)
best_params_df

In [ ]:
import numpy as np
import pandas as pd

# Suppose you have multiple seeds in `results`:
dfs = []
for seed_idx, res_seed in enumerate(results):
    hist = res_seed["history"]
    df = pd.DataFrame(hist)
    df["seed"] = seed_idx  # add a seed column
    dfs.append(df)

history_df = pd.concat(dfs, ignore_index=True)

n_params = len(p_min)  # 9
seeds = history_df["seed"].unique()  # get all unique seeds
colors = plt.cm.tab10(np.linspace(0, 1, len(seeds)))  # assign colors

n_cols = 3
n_rows = int(np.ceil(n_params / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3 * n_rows))
axes = axes.flatten()  # flatten for easy indexing

for i in range(n_params):
    ax = axes[i]

    for seed, color in zip(seeds, colors):
        theta_array = np.stack(history_df.loc[history_df["seed"] == seed, "theta_phys"].values)
        obj_array = history_df.loc[history_df["seed"] == seed, "objective"].values
        ax.scatter(theta_array[:, i], obj_array, label=f"Seed {seed}", color=color, alpha=0.3, s=10)
        ax.set_xlabel(parameter_names[i])

    ax.set_ylabel("Objective")
    # ax.set_title()
    ax.set_xlim(p_min[i], p_max[i])
    ax.grid(True)

# remove empty axes
for j in range(n_params, len(axes)):
    fig.delaxes(axes[j])


# axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Parameter columns
param_cols = ["Imax", "Ce", "Sumax", "Beta", "Pmax", "Tlag", "Kf", "Ks", "FM"]

# Scale the best parameters
scaled_params_df = pd.DataFrame(scale(best_params_df[param_cols].to_numpy()), columns=param_cols)


plt.figure(figsize=(12, 4))
# sns.boxplot(data=scaled_params_df, color = 'lightgray')
# sns.stripplot(data=scaled_params_df, color='black', jitter=False, size=8)
sns.boxplot(
    data=scaled_params_df, color="lightgray", fliersize=0
)  # light gray boxes, hide default outliers
sns.stripplot(data=scaled_params_df, color="black", jitter=True, size=6, marker="o")  # black dots

plt.xticks(rotation=45)
plt.ylabel("Scaled parameter value (0-1)")
plt.title("Scaled best parameter sets across seeds")
# add horizontal grid lines
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()